# YOLO Pipeline Resolution Artifact Investigation

**Goal:** Pin down why `ss=256` produces orientation spikes at ~135° while `ss=512` produces spikes at ~90°/180°.

## Background
Parts 1–7 of `test_orientation_bug.ipynb` established:
- The orientation bug comes from staircase artifacts in YOLOv8's binary masks
- `downscale_pred=False` barely helps — the 1024×1024 mask itself is poor quality
- At `ss=512`: spikes at ~90°/180°
- At `ss=256`: spikes shift to ~135°
- **Part 7 null result:** rasterizing clean ellipses directly at 1024×1024 does NOT reproduce the artifact

The missing piece: the actual YOLO mask pipeline is
`prototype (256×256) → bilinear upsample 4× → threshold ≥0.5 → [INTER_AREA downscale → >0]`

Part 7 never simulated this round-trip. This notebook does.

## Key hypothesis
The bilinear upsample creates smooth diagonal transitions at ellipse boundary corners.
After threshold, these diagonals are real pixels in the 1024 mask.
The INTER_AREA+>0 downscale back to 256 (a 4×4 max-pool) preserves and amplifies them.
The 256 output mask ends up with MORE diagonal boundary pixels than the original prototype —
biasing fitEllipse toward ~135°.

In [ ]:
import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from shapely import segmentize

from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

# ── Synthetic ellipse helpers ──────────────────────────────────────────────

def make_ellipse_polygon(a, b, theta_degrees, cx=0.0, cy=0.0, n_pts=400):
    theta_rad = np.radians(theta_degrees)
    t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
    x = a * np.cos(t);  y = b * np.sin(t)
    x_rot = x * np.cos(theta_rad) - y * np.sin(theta_rad) + cx
    y_rot = x * np.sin(theta_rad) + y * np.cos(theta_rad) + cy
    return Polygon(np.column_stack([x_rot, y_rot]))


def rasterize_ellipse_float(a_px, b_px, theta_degrees, img_size):
    mask = np.zeros((img_size, img_size), dtype=np.float32)
    cx, cy = img_size // 2, img_size // 2
    cv2.ellipse(mask, (cx, cy), (max(1, int(a_px)), max(1, int(b_px))),
                float(theta_degrees), 0, 360, 1.0, -1)
    return mask


def rasterize_ellipse_binary(a_px, b_px, theta_degrees, img_size):
    return (rasterize_ellipse_float(a_px, b_px, theta_degrees, img_size) > 0.5).astype(np.uint8)


def mask_to_poly(mask_uint8):
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea).squeeze(1)
    return Polygon(contour.astype(float)) if len(contour) >= 3 else None


def run_orientation(poly, res=1.0):
    if poly is None or poly.is_empty:
        return None, None, None
    row = pd.Series({"geometry": segmentize(poly, res)})
    try:
        ellipse_poly, a_fit, b_fit, theta_rad = fitEllipse(row)
        theta_deg = np.degrees(theta_rad) % 180
    except Exception:
        return None, None, None
    try:
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        (_, _, la, sa, _, _, _, angle180) = boulder_row(mrr_row)
        aspect = la / sa if sa > 0 else None
    except Exception:
        return None, None, None
    return theta_deg, angle180, aspect

In [ ]:
# ── YOLO mask pipeline simulation ─────────────────────────────────────────
# Actual pipeline in YOLOv8fastv2:
#   1. masks[i]: float soft mask at inference_size (1024), from prototype -> bilinear upsample
#   2. >= 0.5   -> binary float {0,1} at 1024
#   3. if downscale_pred: INTER_AREA to slice_size
#   4. > 0      -> binary uint8

INFERENCE_SIZE = 1024
PROTO_SIZE     = 256   # imgsz / 4 = 256  (confirmed by mask_ratio=1 in training.py)

A_1024, B_1024    = 300, 150        # test ellipse semi-axes at inference_size scale (a/b = 2)
DISSECT_ANGLES    = [0, 30, 45, 60, 90, 120, 135, 150]


def simulate_proto_mask(a_1024, b_1024, theta_deg,
                        proto_size=PROTO_SIZE, inference_size=INFERENCE_SIZE):
    scale   = proto_size / inference_size
    a_proto = max(1.0, a_1024 * scale)
    b_proto = max(1.0, b_1024 * scale)
    mask_proto = rasterize_ellipse_float(a_proto, b_proto, theta_deg, proto_size)
    mask_up    = cv2.resize(mask_proto, (inference_size, inference_size),
                            interpolation=cv2.INTER_LINEAR)
    return (mask_up >= 0.5).astype(np.uint8)


def apply_downscale(mask_1024_uint8, slice_size, threshold=0.0):
    if slice_size == mask_1024_uint8.shape[0]:
        return mask_1024_uint8.copy()
    ds = cv2.resize(mask_1024_uint8.astype(np.float32),
                    (slice_size, slice_size), interpolation=cv2.INTER_AREA)
    return (ds > threshold).astype(np.uint8)


def pipeline_configs(a_1024, b_1024, theta_deg, slice_sizes=(256, 512, 1024)):
    configs = {}
    for ss in slice_sizes:
        scale = ss / INFERENCE_SIZE
        configs["direct_{}".format(ss)] = rasterize_ellipse_binary(
            a_1024 * scale, b_1024 * scale, theta_deg, ss)
    mask_1024 = simulate_proto_mask(a_1024, b_1024, theta_deg)
    configs["yolo_1024"] = mask_1024
    for ss in [s for s in slice_sizes if s < INFERENCE_SIZE]:
        configs["yolo_{}_gt0".format(ss)]  = apply_downscale(mask_1024, ss, threshold=0.0)
        configs["yolo_{}_ge05".format(ss)] = apply_downscale(mask_1024, ss, threshold=0.5)
    return configs

## Part 1 — Visualize pipeline stages

For each test angle show the binary mask at each pipeline stage.
The recovered orientation (red annotation) should reveal where the 135° bias appears.

In [ ]:
stage_labels = [
    ("direct_1024",   "Direct 1024\n(clean baseline)"),
    ("yolo_1024",     "YOLO proto->1024\n(>=0.5 threshold)"),
    ("direct_256",    "Direct 256\n(clean 256)"),
    ("yolo_256_gt0",  "YOLO->256 (>0)\n[production ss=256]"),
    ("yolo_256_ge05", "YOLO->256 (>=0.5)\n[fixed threshold]"),
    ("direct_512",    "Direct 512\n(clean 512)"),
    ("yolo_512_gt0",  "YOLO->512 (>0)\n[production ss=512]"),
]

n_angles = len(DISSECT_ANGLES)
n_stages = len(stage_labels)
fig, axes = plt.subplots(n_stages, n_angles, figsize=(2.2 * n_angles, 2.5 * n_stages))

for col, theta in enumerate(DISSECT_ANGLES):
    cfgs = pipeline_configs(A_1024, B_1024, theta)
    for row_i, (key, label) in enumerate(stage_labels):
        ax = axes[row_i, col]
        mask = cfgs.get(key)
        if mask is not None:
            ax.imshow(mask, cmap="gray", interpolation="nearest", origin="lower")
            theta_rec, _, _ = run_orientation(mask_to_poly(mask), res=1.0)
            ann = "->{:.0f}deg".format(theta_rec) if theta_rec is not None else "fail"
            ax.set_title(ann, fontsize=7, color="red")
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(label, fontsize=7)
        if row_i == 0:
            ax.set_xlabel("theta={}deg".format(theta), fontsize=7, labelpad=1)

plt.suptitle("Mask at each pipeline stage  |  red annotation = recovered orientation", fontsize=10)
plt.tight_layout()
plt.savefig("resolution_part1_stage_masks.png", dpi=120)
plt.show()

## Part 2 — Orientation recovery sweep across all angles

Run 36 test angles (0–175° in 5° steps) through each configuration and plot
recovered theta vs true theta. Direct baselines should be a perfect diagonal.
Production paths should show the characteristic spike locations.

In [ ]:
TEST_ANGLES = list(range(0, 180, 5))

sweep_keys = [
    ("direct_1024",   "Direct 1024 (baseline)"),
    ("yolo_1024",     "YOLO proto->1024"),
    ("direct_512",    "Direct 512"),
    ("yolo_512_gt0",  "YOLO->512 (>0) [prod]"),
    ("yolo_512_ge05", "YOLO->512 (>=0.5)"),
    ("direct_256",    "Direct 256"),
    ("yolo_256_gt0",  "YOLO->256 (>0) [prod]"),
    ("yolo_256_ge05", "YOLO->256 (>=0.5)"),
]

sweep_results = {key: [] for key, _ in sweep_keys}

for theta in TEST_ANGLES:
    cfgs = pipeline_configs(A_1024, B_1024, theta, slice_sizes=(256, 512, 1024))
    for key, _ in sweep_keys:
        mask = cfgs.get(key)
        if mask is not None:
            theta_rec, _, _ = run_orientation(mask_to_poly(mask), res=1.0)
        else:
            theta_rec = None
        sweep_results[key].append(theta_rec)

sweep_df = pd.DataFrame({"true_theta": TEST_ANGLES,
                         **{k: v for k, v in sweep_results.items()}})

for key, _ in sweep_keys:
    errs = []
    for r, t in zip(sweep_df[key], sweep_df["true_theta"]):
        if r is None or (isinstance(r, float) and np.isnan(r)):
            errs.append(np.nan)
        else:
            d = abs(r - t) % 180
            errs.append(min(d, 180 - d))
    sweep_df["{}_err".format(key)] = errs

print("Mean absolute orientation error (degrees):\n")
for key, label in sweep_keys:
    mae = sweep_df["{}_err".format(key)].mean()
    print("  {:<35}  {:5.1f} deg".format(label, mae))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (key, label) in zip(axes, sweep_keys):
    recovered = sweep_df[key].values
    errors    = sweep_df["{}_err".format(key)].values
    ax.plot(TEST_ANGLES, TEST_ANGLES, "k--", lw=1.5, alpha=0.6)
    sc = ax.scatter(TEST_ANGLES, recovered, c=errors, cmap="RdYlGn_r", vmin=0, vmax=45,
                    s=50, zorder=3)
    mae_val = np.nanmean(errors)
    ax.set_title("{}\nMAE = {:.1f} deg".format(label, mae_val), fontsize=9)
    ax.set_xlabel("True theta (deg)")
    ax.set_ylabel("Recovered theta (deg)")
    ax.set_xlim(-5, 185); ax.set_ylim(-5, 185)
    ax.grid(True, alpha=0.3)

plt.colorbar(sc, ax=axes[-1], label="|error| (deg)")
plt.suptitle(
    "Recovered vs true orientation  |  green=accurate, red=45deg error\n"
    "Direct baselines should be perfectly green; YOLO->256 should cluster near 135deg",
    fontsize=11)
plt.tight_layout()
plt.savefig("resolution_part2_angle_sweep.png", dpi=150)
plt.show()

## Part 3 — Stage-by-stage dissection (ss=256 path)

Break the ss=256 production path into individual steps to find exactly which one
introduces the diagonal bias. The stage where MAE first jumps is the cause.

```
Stage 1: rasterize at proto (256)          -> binary at 256
Stage 2: bilinear upsample 4x to 1024      -> float at 1024
Stage 3: threshold >= 0.5                  -> binary at 1024  (= masks[i] from YOLO)
Stage 4: INTER_AREA downscale to 256       -> float average in [0,1]
Stage 5a: > 0   (production)               -> binary at 256
Stage 5b: >= 0.5 (proposed fix)            -> binary at 256
```

In [ ]:
def stage_masks(a_1024, b_1024, theta_deg):
    scale_256 = PROTO_SIZE / INFERENCE_SIZE

    s1 = rasterize_ellipse_binary(a_1024 * scale_256, b_1024 * scale_256,
                                   theta_deg, PROTO_SIZE)

    s2_float = cv2.resize(s1.astype(np.float32), (INFERENCE_SIZE, INFERENCE_SIZE),
                           interpolation=cv2.INTER_LINEAR)

    s3 = (s2_float >= 0.5).astype(np.uint8)

    s4_float = cv2.resize(s3.astype(np.float32), (PROTO_SIZE, PROTO_SIZE),
                           interpolation=cv2.INTER_AREA)

    s5a = (s4_float > 0.0).astype(np.uint8)
    s5b = (s4_float >= 0.5).astype(np.uint8)

    added   = np.clip(s5a.astype(int) - s1.astype(int), 0, 1).astype(np.uint8)
    removed = np.clip(s1.astype(int)  - s5a.astype(int), 0, 1).astype(np.uint8)

    return {
        "s1_proto_binary":   s1,
        "s3_yolo_1024":      s3,
        "s4_area_float":     s4_float,
        "s5a_yolo_256_gt0":  s5a,
        "s5b_yolo_256_ge05": s5b,
        "diff_added":        added,
        "diff_removed":      removed,
    }

## Part 4 — Quantify orientation error at each stage

For all 36 test angles compute MAE at each stage. The step where MAE first jumps
toward 135° pinpoints the culprit.

In [ ]:
stage_error_keys = [
    ("s1_proto_binary",   "Stage 1: proto binary (256)"),
    ("s3_yolo_1024",      "Stage 3: YOLO 1024"),
    ("s5a_yolo_256_gt0",  "YOLO->256 >0 [prod]"),
    ("s5b_yolo_256_ge05", "YOLO->256 >=0.5 [fix]"),
]

stage_err_results = {k: [] for k, _ in stage_error_keys}
stage_err_results["yolo_512_gt0"]  = []
stage_err_results["yolo_512_ge05"] = []

for theta in TEST_ANGLES:
    stages = stage_masks(A_1024, B_1024, theta)
    cfgs   = pipeline_configs(A_1024, B_1024, theta)
    for key, _ in stage_error_keys:
        poly = mask_to_poly(stages[key])
        theta_rec, _, _ = run_orientation(poly, res=1.0)
        stage_err_results[key].append(theta_rec)
    for key in ["yolo_512_gt0", "yolo_512_ge05"]:
        poly = mask_to_poly(cfgs[key])
        theta_rec, _, _ = run_orientation(poly, res=1.0)
        stage_err_results[key].append(theta_rec)

stage_df = pd.DataFrame({"true_theta": TEST_ANGLES, **stage_err_results})

for col in stage_err_results:
    errs = []
    for r, t in zip(stage_df[col], stage_df["true_theta"]):
        if r is None or (isinstance(r, float) and np.isnan(r)):
            errs.append(np.nan)
        else:
            d = abs(r - t) % 180
            errs.append(min(d, 180 - d))
    stage_df["{}_err".format(col)] = errs

print("MAE at each pipeline stage:\n")
for col in stage_err_results:
    print("  {:<30}  {:5.1f} deg".format(col, stage_df["{}_err".format(col)].mean()))

In [ ]:
stage_plot_configs = [
    ("s1_proto_binary",   "Stage 1: proto binary (256)"),
    ("s3_yolo_1024",      "Stage 3: YOLO 1024"),
    ("s5a_yolo_256_gt0",  "YOLO->256 >0 [prod]"),
    ("s5b_yolo_256_ge05", "YOLO->256 >=0.5 [fix]"),
    ("yolo_512_gt0",      "YOLO->512 >0 [prod]"),
    ("yolo_512_ge05",     "YOLO->512 >=0.5 [fix]"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (key, label) in zip(axes, stage_plot_configs):
    err_col   = "{}_err".format(key)
    recovered = stage_df[key].values
    errors    = stage_df[err_col].values if err_col in stage_df.columns else np.zeros(len(TEST_ANGLES))
    ax.plot(TEST_ANGLES, TEST_ANGLES, "k--", lw=1.5, alpha=0.5)
    sc = ax.scatter(TEST_ANGLES, recovered, c=errors, cmap="RdYlGn_r", vmin=0, vmax=45,
                    s=50, zorder=3)
    mae_val = np.nanmean(errors)
    ax.set_title("{}\nMAE = {:.1f} deg".format(label, mae_val), fontsize=9)
    ax.set_xlabel("True theta (deg)")
    ax.set_ylabel("Recovered theta (deg)")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5, 185); ax.set_ylim(-5, 185)

plt.colorbar(sc, ax=axes[-1], label="|error| (deg)")
plt.suptitle("Orientation recovery per pipeline stage -- which step introduces the 135deg bias?",
             fontsize=11)
plt.tight_layout()
plt.savefig("resolution_part4_per_stage_error.png", dpi=150)
plt.show()

## Part 5 — Proto size sensitivity

`mask_ratio=1` fixes proto_size = imgsz/4 = 256. But what if the model used a different
prototype resolution (e.g., ultralytics default 160)? Does the spike location shift?

In [ ]:
PROTO_SIZES_TO_TEST = [64, 128, 160, 256, 512]

proto_sweep = {}

for ps in PROTO_SIZES_TO_TEST:
    for ss in [256, 512]:
        for thresh, tlabel in [(0.0, "gt0"), (0.5, "ge05")]:
            rec_list = []
            for theta in TEST_ANGLES:
                scale   = ps / INFERENCE_SIZE
                a_proto = max(1.0, A_1024 * scale)
                b_proto = max(1.0, B_1024 * scale)
                mp  = rasterize_ellipse_float(a_proto, b_proto, theta, ps)
                mu  = cv2.resize(mp, (INFERENCE_SIZE, INFERENCE_SIZE),
                                 interpolation=cv2.INTER_LINEAR)
                m1  = (mu >= 0.5).astype(np.uint8)
                mss = apply_downscale(m1, ss, threshold=thresh)
                poly = mask_to_poly(mss)
                theta_rec, _, _ = run_orientation(poly, res=1.0)
                rec_list.append(theta_rec)
            proto_sweep[(ps, ss, tlabel)] = rec_list

rows = []
for (ps, ss, tlabel), rec_list in proto_sweep.items():
    errs = []
    for r, t in zip(rec_list, TEST_ANGLES):
        if r is None or (isinstance(r, float) and np.isnan(r)):
            errs.append(np.nan)
        else:
            d = abs(r - t) % 180
            errs.append(min(d, 180 - d))
    rows.append({"proto_size": ps, "slice_size": ss, "threshold": tlabel,
                 "mae": np.nanmean(errs), "max_err": np.nanmax(errs)})

proto_mae_df = pd.DataFrame(rows).sort_values(["slice_size", "proto_size"])
print(proto_mae_df.to_string(index=False))

In [ ]:
for ss in [256, 512]:
    subset = proto_mae_df[proto_mae_df["slice_size"] == ss]
    pivot  = subset.pivot(index="proto_size", columns="threshold", values="mae")
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=30)
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)));   ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Threshold after INTER_AREA downscale")
    ax.set_ylabel("Proto size (px)")
    ax.set_title("MAE (deg) -- slice_size={}  (green=accurate, red=poor)".format(ss))
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, "{:.1f}".format(pivot.values[i, j]),
                    ha="center", va="center", fontsize=9)
    plt.colorbar(im, ax=ax, label="MAE (deg)")
    plt.tight_layout()
    plt.savefig("resolution_part5_proto_sweep_ss{}.png".format(ss), dpi=150)
    plt.show()

## Part 6 — Diagonal bias anatomy

Visualize WHERE on the ellipse boundary the round-trip adds and removes pixels.
If added pixels (red) concentrate at the ellipse corners (the curved turning points),
that confirms the mechanism: bilinear upsample rounds the corners diagonally,
the max-pool preserves those diagonal pixels, biasing fitEllipse toward ~135°.

In [ ]:
n_ang = len(DISSECT_ANGLES)
fig, axes = plt.subplots(3, n_ang, figsize=(2.5 * n_ang, 8))

for col, theta in enumerate(DISSECT_ANGLES):
    stages  = stage_masks(A_1024, B_1024, theta)
    s1      = stages["s1_proto_binary"]
    s5a     = stages["s5a_yolo_256_gt0"]
    added   = stages["diff_added"]
    removed = stages["diff_removed"]

    # Row 0: proto binary
    ax = axes[0, col]
    ax.imshow(s1, cmap="gray", interpolation="nearest", origin="lower")
    t1, _, _ = run_orientation(mask_to_poly(s1), res=1.0)
    ttl = "theta={}\n->{:.0f}".format(theta, t1) if t1 is not None else "theta={}\nfail".format(theta)
    ax.set_title(ttl, fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
    if col == 0:
        ax.set_ylabel("Stage 1\n(proto binary)", fontsize=7)

    # Row 1: production output with added/removed pixels highlighted
    ax = axes[1, col]
    rgb = np.stack([s5a, s5a, s5a], axis=-1).astype(np.float32)
    rgb[added   > 0] = [1.0, 0.0, 0.0]
    rgb[removed > 0] = [0.0, 0.0, 1.0]
    ax.imshow(rgb, interpolation="nearest", origin="lower")
    t5a, _, _ = run_orientation(mask_to_poly(s5a), res=1.0)
    ttl5 = "->{:.0f}".format(t5a) if t5a is not None else "fail"
    ax.set_title(ttl5, fontsize=7, color="darkred")
    ax.set_xticks([]); ax.set_yticks([])
    if col == 0:
        ax.set_ylabel("Stage 5a (prod)\nred=added, blue=removed", fontsize=7)

    # Row 2: boundary-only view
    ax = axes[2, col]
    kernel   = np.ones((3, 3), np.uint8)
    s1_edge  = s1 - cv2.erode(s1, kernel, iterations=1)
    overlay  = np.zeros((*s1.shape, 3), dtype=np.float32)
    overlay[s1_edge > 0] = [0.7, 0.7, 0.7]
    overlay[added   > 0] = [1.0, 0.0, 0.0]
    overlay[removed > 0] = [0.0, 0.0, 1.0]
    ax.imshow(overlay, interpolation="nearest", origin="lower")
    ax.set_title("+{} / -{} px".format(added.sum(), removed.sum()), fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])
    if col == 0:
        ax.set_ylabel("Boundary diff\n(gray=orig, r=+, b=-)", fontsize=7)

plt.suptitle(
    "Round-trip artifact anatomy (proto->1024->256)\n"
    "Red = pixels ADDED, Blue = REMOVED by upsample+downscale.\n"
    "Red at corners confirms the diagonal bias mechanism.",
    fontsize=9)
plt.tight_layout()
plt.savefig("resolution_part6_boundary_anatomy.png", dpi=120)
plt.show()

## Summary

### Parts 1–6 result: all configurations clean (null result)
All scatter plots were diagonal and fully green for `A_1024=300, B_1024=150`.
This rules out the pipeline geometry alone as the cause — the bilinear upsample +
threshold + INTER_AREA max-pool does not introduce orientation errors for a
well-resolved synthetic ellipse.

**Revised hypothesis:** The artifact requires *small* ellipses with few pixels at
prototype resolution. Real boulders near the area threshold occupy only a handful
of prototype pixels; at that scale the rasterized boundary is dominated by a coarse
staircase that the downstream pipeline cannot recover from.

### Part 7
Tests whether shrinking the ellipse reproduces the real-data spikes.
Expected: large sizes stay green; small sizes (a<~15px at inference_size) show
spikes at ~90°/180° (ss=512) and ~135° (ss=256), matching the real data pattern.

**Conclusion:** [fill in after running Part 7]

## Part 7 — Size sweep: does a small ellipse reproduce the real-data spikes?

Parts 1–6 used `A_1024=300, B_1024=150` (proto axes: 75×37 px) — all green, no artifacts.
Real boulders near the area threshold at 0.48 m/px occupy ~5–15 inference pixels
across, meaning only ~1–4 proto pixels. At that scale the rasterized boundary is a
coarse staircase that EllipseModel cannot resolve.

**Prediction:**
- Large sizes → green (Parts 1–6 result)
- Small sizes (a~8–15 at inference_size) → spikes emerge
- The transition point quantifies the minimum reliable-orientation boulder size

Proto-pixel equivalents for each size (proto_size=256, inference_size=1024, scale=0.25):
```
a=300  -> proto a=75 px   (well-resolved)
a=80   -> proto a=20 px
a=30   -> proto a=7.5 px
a=15   -> proto a=3.75 px (near threshold for ss=256)
a=8    -> proto a=2 px    (near threshold for ss=512)
```

In [ ]:
# (a, b) semi-axes in inference_size=1024 coordinates, label, approx real-world semi-axis
# at Moon resolution 0.48 m/px with ss=512 (inference pixel = 0.24 m)
SIZE_SWEEP = [
    (300, 150, "large   a=300 (~72m)"),
    (80,  50,  "medium  a=80  (~19m)"),
    (30,  20,  "small   a=30  (~7m) "),
    (15,  10,  "v.small a=15  (~4m) "),
    (8,   5,   "tiny    a=8   (~2m) "),
]

size_sweep_results = {}

for a_1024, b_1024, size_label in SIZE_SWEEP:
    for ss_key in ["yolo_256_gt0", "yolo_512_gt0"]:
        rec_list = []
        for theta in TEST_ANGLES:
            cfgs = pipeline_configs(a_1024, b_1024, theta)
            mask = cfgs.get(ss_key)
            if mask is not None and mask.sum() > 0:
                theta_rec, _, _ = run_orientation(mask_to_poly(mask), res=1.0)
            else:
                theta_rec = None
            rec_list.append(theta_rec)

        errs = []
        for r, t in zip(rec_list, TEST_ANGLES):
            if r is None or (isinstance(r, float) and np.isnan(r)):
                errs.append(np.nan)
            else:
                d = abs(r - t) % 180
                errs.append(min(d, 180 - d))

        size_sweep_results[(size_label, ss_key)] = {
            "recovered": rec_list,
            "error":     errs,
            "mae":       np.nanmean(errs),
            "n_fail":    sum(1 for r in rec_list if r is None),
        }

    a_proto = a_1024 * PROTO_SIZE / INFERENCE_SIZE
    b_proto = b_1024 * PROTO_SIZE / INFERENCE_SIZE
    mae_256 = size_sweep_results[(size_label, "yolo_256_gt0")]["mae"]
    mae_512 = size_sweep_results[(size_label, "yolo_512_gt0")]["mae"]
    print("  {}  proto={:.1f}x{:.1f}px  MAE_256={:.1f}deg  MAE_512={:.1f}deg".format(
        size_label, a_proto, b_proto, mae_256, mae_512))

In [ ]:
n_sizes = len(SIZE_SWEEP)
fig, axes = plt.subplots(n_sizes, 2, figsize=(12, 3.5 * n_sizes), sharex=True, sharey=True)

ss_configs = [
    ("yolo_256_gt0", "YOLO->256 >0  [prod ss=256]"),
    ("yolo_512_gt0", "YOLO->512 >0  [prod ss=512]"),
]

for row_i, (a_1024, b_1024, size_label) in enumerate(SIZE_SWEEP):
    a_proto = a_1024 * PROTO_SIZE / INFERENCE_SIZE
    b_proto = b_1024 * PROTO_SIZE / INFERENCE_SIZE
    for col_i, (ss_key, ss_title) in enumerate(ss_configs):
        ax      = axes[row_i, col_i]
        res     = size_sweep_results[(size_label, ss_key)]
        errors  = res["error"]
        mae_val = res["mae"]
        n_fail  = res["n_fail"]

        ax.plot(TEST_ANGLES, TEST_ANGLES, "k--", lw=1.5, alpha=0.5)
        sc = ax.scatter(TEST_ANGLES, res["recovered"], c=errors, cmap="RdYlGn_r",
                        vmin=0, vmax=45, s=50, zorder=3)
        ax.set_title(
            "{}  |  {}
proto {:.1f}x{:.1f}px  MAE={:.1f}deg  fails={}".format(
                size_label, ss_title, a_proto, b_proto, mae_val, n_fail),
            fontsize=8)
        ax.set_xlabel("True theta (deg)")
        ax.set_ylabel("Recovered theta (deg)")
        ax.set_xlim(-5, 185); ax.set_ylim(-5, 185)
        ax.grid(True, alpha=0.3)

plt.colorbar(sc, ax=axes[-1, -1], label="|error| (deg)")
plt.suptitle(
    "Part 7: Size sweep
"
    "Large ellipses (top rows) should be green; small ellipses (bottom rows) should
"
    "show spikes at 90/180deg (ss=512) or ~135deg (ss=256) matching real data.",
    fontsize=11)
plt.tight_layout()
plt.savefig("resolution_part7_size_sweep.png", dpi=150)
plt.show()